In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 12)
plt.rcParams['font.size'] = 10

# =====================================================
# Load All Scenario Results (FREQUENCY ELASTICITY)
# =====================================================

# Base directory for FREQUENCY elasticity sensitivity results
BASE_DIR = Path("../Results/Sensitivity analysis/Frequency/Frequency_Output")

# Load all scenarios
scenarios = {}
freq_elasticity_levels = [10, 20, 30, 40, 50]

for level in freq_elasticity_levels:
    try:
        master = pd.read_excel(BASE_DIR / f"master_solution_decisions_{level}_percent.xlsx")
        convergence = pd.read_excel(BASE_DIR / f"benders_convergence_{level}_percent.xlsx")
        scenarios[level] = {
            'master': master,
            'convergence': convergence
        }
        print(f"✅ Loaded {level}% frequency elasticity scenario")
    except FileNotFoundError:
        print(f"⚠️  Missing files for {level}% scenario")


fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Modal Split Evolution with Increasing Frequency Elasticity', 
             fontsize=16, fontweight='bold', y=0.995)

for idx, level in enumerate(freq_elasticity_levels):
    ax = axes[idx // 3, idx % 3]
    
    master = scenarios[level]['master']
    
    # Calculate modal shares
    mode_totals = master.groupby('mode')['TEU_y'].sum()
    
    # Create pie chart
    colors = ['#3498db', '#e74c3c', '#95a5a6']
    explode = (0.05, 0, 0) if 'Intermodal' in mode_totals.index else (0, 0, 0)
    
    wedges, texts, autotexts = ax.pie(
        mode_totals.values,
        labels=mode_totals.index,
        autopct='%1.1f%%',
        colors=colors[:len(mode_totals)],
        startangle=90,
        explode=explode[:len(mode_totals)],
        textprops={'fontsize': 10, 'weight': 'bold'}
    )
    
    # Make percentage text white and bold
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    
    ax.set_title(f'+{level}% Frequency Elasticity\nTotal: {mode_totals.sum():.0f} TEU/week', 
                 fontsize=12, fontweight='bold')

# Hide the 6th subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig(BASE_DIR / 'graph_1_modal_split_evolution_freq.png', dpi=300, bbox_inches='tight')
print("✅ Saved: graph_1_modal_split_evolution_freq.png")
plt.close()

# =====================================================
# Figure 2: Intermodal Market Share Trend
# =====================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Impact of Frequency Elasticity on Intermodal Competitiveness', 
             fontsize=16, fontweight='bold')

# Calculate metrics for all scenarios
intermodal_shares = []
avg_prices = []
total_volumes = []

for level in freq_elasticity_levels:
    master = scenarios[level]['master']
    
    # Modal share
    mode_totals = master.groupby('mode')['TEU_y'].sum()
    total = mode_totals.sum()
    im_share = (mode_totals.get('Intermodal', 0) / total * 100) if total > 0 else 0
    intermodal_shares.append(im_share)
    
    # Average intermodal price
    im_data = master[master['mode'] == 'Intermodal']
    avg_price = (im_data['price_p'] * im_data['TEU_y']).sum() / im_data['TEU_y'].sum() if im_data['TEU_y'].sum() > 0 else 0
    avg_prices.append(avg_price)
    
    # Total intermodal volume
    total_volumes.append(mode_totals.get('Intermodal', 0))

# Plot 1: Market share trend
ax1.plot(freq_elasticity_levels, intermodal_shares, 'o-', linewidth=2.5, markersize=10,
         color='#3498db', markerfacecolor='white', markeredgewidth=2)
ax1.fill_between(freq_elasticity_levels, 0, intermodal_shares, alpha=0.2, color='#3498db')
ax1.set_xlabel('Increase in Frequency Elasticity (%)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Intermodal Market Share (%)', fontsize=12, fontweight='bold')
ax1.set_title('Intermodal Market Share vs. Frequency Sensitivity', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_ylim([0, max(intermodal_shares) * 1.2 if intermodal_shares else 1])

# Add value labels
for x, y in zip(freq_elasticity_levels, intermodal_shares):
    ax1.annotate(f'{y:.1f}%', xy=(x, y), xytext=(0, 10), 
                 textcoords='offset points', ha='center', fontsize=9, fontweight='bold')

# Plot 2: Price vs Volume relationship (under frequency elasticity changes)
scatter = ax2.scatter(avg_prices, total_volumes, s=300, c=freq_elasticity_levels, 
                      cmap='RdYlGn_r', edgecolors='black', linewidth=2, alpha=0.8)

# Add labels for each point
for i, level in enumerate(freq_elasticity_levels):
    ax2.annotate(f'+{level}%', xy=(avg_prices[i], total_volumes[i]), 
                 xytext=(8, 8), textcoords='offset points', 
                 fontsize=10, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

ax2.set_xlabel('Average Intermodal Price (CHF/TEU)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Total Intermodal Volume (TEU/week)', fontsize=12, fontweight='bold')
ax2.set_title('Price-Volume Trade-off under Frequency Elasticity Changes', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, linestyle='--')

cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Frequency Elasticity Increase (%)', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(BASE_DIR / 'graph_2_intermodal_trends_freq.png', dpi=300, bbox_inches='tight')
print("✅ Saved: graph_2_intermodal_trends_freq.png")
plt.close()

# =====================================================
# Figure 3: OD-Level Price Adjustments
# =====================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('OD-Level Strategic Price Adjustments by Frequency Elasticity Scenario', 
             fontsize=16, fontweight='bold', y=0.995)

# Compare specific elasticity levels
comparison_pairs = [(10, 30), (10, 50), (20, 40), (30, 50)]

for idx, (level1, level2) in enumerate(comparison_pairs):
    ax = axes[idx // 2, idx % 2]
    
    master1 = scenarios[level1]['master']
    master2 = scenarios[level2]['master']
    
    # Filter intermodal only
    im1 = master1[master1['mode'] == 'Intermodal'].copy()
    im2 = master2[master2['mode'] == 'Intermodal'].copy()
    
    # Create OD identifier
    im1['od'] = im1['origin'] + '-' + im1['destination']
    im2['od'] = im2['origin'] + '-' + im2['destination']
    
    # Merge on OD
    comparison = im1[['od', 'price_p', 'TEU_y']].merge(
        im2[['od', 'price_p', 'TEU_y']],
        on='od',
        suffixes=(f'_{level1}', f'_{level2}')
    )
    
    # Calculate price change percentage
    comparison['price_change_pct'] = ((comparison[f'price_p_{level2}'] - comparison[f'price_p_{level1}']) / 
                                      comparison[f'price_p_{level1}']) * 100
    
    # Color by volume in second scenario
    scatter = ax.scatter(comparison[f'price_p_{level1}'], 
                         comparison[f'price_p_{level2}'],
                         s=comparison[f'TEU_y_{level2}'] * 3,
                         c=comparison['price_change_pct'],
                         cmap='RdYlGn_r',
                         alpha=0.7,
                         edgecolors='black',
                         linewidth=0.5)
    
    # Add diagonal line
    max_price = max(comparison[f'price_p_{level1}'].max(), comparison[f'price_p_{level2}'].max())
    ax.plot([0, max_price], [0, max_price], 'k--', linewidth=1, alpha=0.5, label='No Change')
    
    ax.set_xlabel(f'Price at +{level1}% Freq Elasticity (CHF/TEU)', fontsize=11, fontweight='bold')
    ax.set_ylabel(f'Price at +{level2}% Freq Elasticity (CHF/TEU)', fontsize=11, fontweight='bold')
    ax.set_title(f'Comparison: +{level1}% vs +{level2}% Frequency Elasticity', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=9)
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Price Change (%)', fontsize=9)

plt.tight_layout()
plt.savefig(BASE_DIR / 'graph_3_od_price_adjustments_freq.png', dpi=300, bbox_inches='tight')
print("✅ Saved: graph_3_od_price_adjustments_freq.png")
plt.close()

# =====================================================
# Figure 4: Frequency Response
# =====================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Service Frequency Adjustments Across Frequency Elasticity Scenarios', 
             fontsize=16, fontweight='bold', y=0.995)

for idx, level in enumerate(freq_elasticity_levels):
    ax = axes[idx // 3, idx % 3]
    
    master = scenarios[level]['master']
    
    # Filter active services (frequency > 0)
    active = master[master['frequency_f'] > 0.01].copy()
    
    # Create OD labels
    active['od_label'] = active['origin'].str[:3] + '-' + active['destination'].str[:3]
    
    # Separate by mode
    im_active = active[active['mode'] == 'Intermodal'].sort_values('frequency_f', ascending=True).tail(10)
    road_active = active[active['mode'] == 'Road'].sort_values('frequency_f', ascending=True).tail(10)
    
    # Plot horizontal bars
    y_pos_im = np.arange(len(im_active))
    y_pos_road = np.arange(len(road_active)) + len(im_active) + 1
    
    ax.barh(y_pos_im, im_active['frequency_f'], color='#3498db', 
            alpha=0.7, edgecolor='black', linewidth=0.5, label='Intermodal')
    ax.barh(y_pos_road, road_active['frequency_f'], color='#e74c3c', 
            alpha=0.7, edgecolor='black', linewidth=0.5, label='Road')
    
    # Set labels
    all_labels = list(im_active['od_label']) + [''] + list(road_active['od_label'])
    all_pos = list(y_pos_im) + [len(im_active)] + list(y_pos_road)
    
    ax.set_yticks(all_pos)
    ax.set_yticklabels(all_labels, fontsize=8)
    ax.set_xlabel('Frequency (dep/week)', fontsize=10, fontweight='bold')
    ax.set_title(f'+{level}% Frequency Elasticity\nTop 10 Routes by Mode', fontsize=11, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(axis='x', alpha=0.3)

# Hide the 6th subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig(BASE_DIR / 'graph_4_frequency_response_freq.png', dpi=300, bbox_inches='tight')
print("✅ Saved: graph_4_frequency_response_freq.png")
plt.close()

# =====================================================
# Figure 5: Revenue and Profit Analysis
# =====================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Financial Impact of Increased Frequency Elasticity', 
             fontsize=16, fontweight='bold', y=0.995)

# Calculate financial metrics
revenues = []
profits = []
im_revenues = []
road_revenues = []

for level in freq_elasticity_levels:
    master = scenarios[level]['master']
    conv = scenarios[level]['convergence']
    
    # Total revenue
    total_rev = (master['price_p'] * master['TEU_y']).sum()
    revenues.append(total_rev)
    
    # Final profit
    final_profit = conv['upper_bound'].iloc[-1]
    profits.append(final_profit)
    
    # Modal revenues
    im_rev = (master[master['mode'] == 'Intermodal']['price_p'] * 
              master[master['mode'] == 'Intermodal']['TEU_y']).sum()
    road_rev = (master[master['mode'] == 'Road']['price_p'] * 
                master[master['mode'] == 'Road']['TEU_y']).sum()
    
    im_revenues.append(im_rev)
    road_revenues.append(road_rev)

# Plot 1: Total Revenue
ax1 = axes[0, 0]
bars = ax1.bar(freq_elasticity_levels, revenues, color='#2ecc71', 
               alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Frequency Elasticity Increase (%)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Total Revenue (CHF/week)', fontsize=11, fontweight='bold')
ax1.set_title('Total Revenue Trend', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:,.0f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# Plot 2: Profit
ax2 = axes[0, 1]
bars = ax2.bar(freq_elasticity_levels, profits, color='#f39c12', 
               alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Frequency Elasticity Increase (%)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Profit (CHF/week)', fontsize=11, fontweight='bold')
ax2.set_title('Profit Trend', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:,.0f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# Plot 3: Stacked revenue by mode
ax3 = axes[1, 0]
x_pos = np.arange(len(freq_elasticity_levels))
width = 0.6

p1 = ax3.bar(x_pos, im_revenues, width, label='Intermodal', 
             color='#3498db', alpha=0.7, edgecolor='black', linewidth=1)
p2 = ax3.bar(x_pos, road_revenues, width, bottom=im_revenues,
             label='Road', color='#e74c3c', alpha=0.7, edgecolor='black', linewidth=1)

ax3.set_xlabel('Frequency Elasticity Increase (%)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Revenue (CHF/week)', fontsize=11, fontweight='bold')
ax3.set_title('Revenue Composition by Mode', fontsize=12, fontweight='bold')
ax3.set_xticks(x_pos)
ax3.set_xticklabels([f'+{l}%' for l in freq_elasticity_levels])
ax3.legend(loc='upper right', fontsize=10)
ax3.grid(axis='y', alpha=0.3)

# Plot 4: Percentage changes from baseline
ax4 = axes[1, 1]
baseline_rev = revenues[0]
baseline_profit = profits[0]

rev_changes = [(r - baseline_rev) / baseline_rev * 100 for r in revenues]
profit_changes = [(p - baseline_profit) / baseline_profit * 100 for p in profits]

x_pos = np.arange(len(freq_elasticity_levels))
width = 0.35

bars1 = ax4.bar(x_pos - width/2, rev_changes, width, label='Revenue Change',
                color='#2ecc71', alpha=0.7, edgecolor='black', linewidth=1)
bars2 = ax4.bar(x_pos + width/2, profit_changes, width, label='Profit Change',
                color='#f39c12', alpha=0.7, edgecolor='black', linewidth=1)

ax4.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax4.set_xlabel('Frequency Elasticity Increase (%)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Change from +10% Baseline (%)', fontsize=11, fontweight='bold')
ax4.set_title('Relative Changes from +10% Baseline (Frequency Elasticity)', fontsize=12, fontweight='bold')
ax4.set_xticks(x_pos)
ax4.set_xticklabels([f'+{l}%' for l in freq_elasticity_levels])
ax4.legend(loc='best', fontsize=10)
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / 'graph_5_financial_analysis_freq.png', dpi=300, bbox_inches='tight')
print("✅ Saved: graph_5_financial_analysis_freq.png")
plt.close()

# =====================================================
# Figure 6: Convergence Comparison
# =====================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Benders Decomposition Convergence Across Frequency Elasticity Scenarios', 
             fontsize=16, fontweight='bold', y=0.995)

for idx, level in enumerate(freq_elasticity_levels):
    ax = axes[idx // 3, idx % 3]
    
    conv = scenarios[level]['convergence']
    
    # Plot bounds
    ax.plot(conv['iteration'], conv['upper_bound'], 
            'b-o', label='Upper Bound', linewidth=2, markersize=4, alpha=0.7)
    ax.plot(conv['iteration'], conv['lower_bound'], 
            'r-s', label='Lower Bound', linewidth=2, markersize=4, alpha=0.7)
    
    # Fill gap
    ax.fill_between(conv['iteration'], 
                    conv['lower_bound'], 
                    conv['upper_bound'],
                    alpha=0.2, color='gray', label='Gap')
    
    # Add final gap annotation
    final_gap = conv['upper_bound'].iloc[-1] - conv['lower_bound'].iloc[-1]
    final_rel_gap = (final_gap / conv['upper_bound'].iloc[-1] * 100) if conv['upper_bound'].iloc[-1] != 0 else 0
    
    ax.text(0.98, 0.02, f'Final Gap: {final_rel_gap:.2f}%\nIterations: {len(conv)}',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel('Iteration', fontsize=10, fontweight='bold')
    ax.set_ylabel('Objective Value (CHF/week)', fontsize=10, fontweight='bold')
    ax.set_title(f'+{level}% Frequency Elasticity', fontsize=11, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide the 6th subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig(BASE_DIR / 'graph_6_convergence_comparison_freq.png', dpi=300, bbox_inches='tight')
print("✅ Saved: graph_6_convergence_comparison_freq.png")
plt.close()

# =====================================================
# Summary Statistics Table
# =====================================================

summary_data = []
for level in freq_elasticity_levels:
    master = scenarios[level]['master']
    conv = scenarios[level]['convergence']
    
    mode_totals = master.groupby('mode')['TEU_y'].sum()
    total_teu = mode_totals.sum()
    im_share = (mode_totals.get('Intermodal', 0) / total_teu * 100) if total_teu > 0 else 0
    
    revenue = (master['price_p'] * master['TEU_y']).sum()
    profit = conv['upper_bound'].iloc[-1]
    iterations = len(conv)
    
    summary_data.append({
        'Frequency_Elasticity_Increase_%': level,
        'Total_TEU': total_teu,
        'Intermodal_Share_%': round(im_share, 2),
        'Total_Revenue_CHF': round(revenue, 2),
        'Profit_CHF': round(profit, 2),
        'Benders_Iterations': iterations
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_excel(BASE_DIR / 'summary_statistics_freq.xlsx', index=False)
print("✅ Saved: summary_statistics_freq.xlsx")

# Print summary to console
print("\n" + "="*80)
print("FREQUENCY ELASTICITY SENSITIVITY ANALYSIS SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

print("\n🎉 All visualizations for FREQUENCY elasticity completed successfully!")
print(f"📁 Results saved to: {BASE_DIR}")
print("\nGenerated files:")
print("  • graph_1_modal_split_evolution_freq.png")
print("  • graph_2_intermodal_trends_freq.png")
print("  • graph_3_od_price_adjustments_freq.png")
print("  • graph_4_frequency_response_freq.png")
print("  • graph_5_financial_analysis_freq.png")
print("  • graph_6_convergence_comparison_freq.png")
print("  • summary_statistics_freq.xlsx")


SyntaxError: invalid syntax (2611727010.py, line 36)